# LIBS

In [ ]:
#Libs
import pandas as pd
import glob
import os

In [1]:
# Caminho da pasta com os dados exportados do GEE
path = '/content/drive/MyDrive/Results_V0'

#Em versões futuras já adicionar o caminho do github com os arquivos

In [ ]:
# Listar todos os arquivos CSV na pasta para leitura
csv_files = glob.glob(os.path.join(path, "*.csv"))

# Vamos concatenar todos os dataframes e aproveitar informações do nome do arquivo para criar novas colunas úteis

In [ ]:
# Lista para armazenar os DataFrames
dfs = []

# Processar cada arquivo
for file in csv_files:
    # Extrair o nome do arquivo sem a extensão
    filename = os.path.basename(file).replace('.csv', '')

    # Dividir o nome do arquivo em partes
    parts = filename.split('_')

    # Garantir que há pelo menos 5 partes para extrair as variáveis
    if len(parts) >= 5:
        emission_source = parts[2]  # Segunda parte
        type_gas = parts[3]  # Terceira parte
        year = parts[4]  # Quarta parte

        # Carregar o CSV
        df = pd.read_csv(file)

        # Adicionar as novas colunas
        df['emission_source'] = emission_source
        df['type_gas'] = type_gas
        df['year'] = year

        # Adicionar à lista
        dfs.append(df)

# Concatenar todos os DataFrames
df_final = pd.concat(dfs, ignore_index=True)

# Exibir as primeiras linhas
df_final.head()

,system:index,AREA_KM2,Bioma,CD_Bioma,CD_UF,FID_BR_UF_,FID_lm_bio,NM_REGIAO,NM_UF,SIGLA_UF,sum,.geo,emission_source,type_gas,year
0,0000000000000000002a,903208.361,Amazônia,1,51,22,0,Centro-oeste,Mato Grosso,MT,486942.639563,"{""type"":""MultiPoint"",""coordinates"":[]}",def,n2o,2017
1,00000000000000000029,329651.496,Amazônia,1,21,5,0,Nordeste,Maranhão,MA,82224.965385,"{""type"":""MultiPoint"",""coordinates"":[]}",def,n2o,2017
2,0000000000000000002b,237754.172,Amazônia,1,11,25,0,Norte,Rondônia,RO,549228.647120,"{""type"":""MultiPoint"",""coordinates"":[]}",def,n2o,2017
3,0000000000000000002d,164173.429,Amazônia,1,12,0,0,Norte,Acre,AC,132647.773896,"{""type"":""MultiPoint"",""coordinates"":[]}",def,n2o,2017
4,0000000000000000002e,1559255.881,Amazônia,1,13,1,0,Norte,Amazonas,AM,561138.889319,"{""type"":""MultiPoint"",""coordinates"":[]}",def,n2o,2017


#Create - tbl - aux 'Correlation factor by biome

In [ ]:
import pandas as pd

# Criando o df_aux com os fatores de correção (fc)
df_aux = pd.DataFrame({
    'Bioma': ["Amazônia", "Caatinga", "Cerrado", "Mata Atlântica", "Pampa", "Pantanal"], #PT
    'Biome': ["Amazon", "Caatinga", "Cerrado", "Atlantic Forest", "Pampa", "Pantanal"], #En
    'fc': [1.013, 1.019, 0.995, 1.000, 0.996, 1.002],
    'fc_positivo': [1.019, 1.024, 1.001, 1.005, 1.009, 1.011],
    'fc_negativo': [1.010, 1.014, 0.989, 0.995, 0.983, 0.993],
    'IC': [0.006, 0.005, 0.006, 0.005, 0.013, 0.009]
})


In [ ]:
# Realizar o merge de df_final com df_aux usando a coluna 'Bioma' como chave
df_final2 = df_final.merge(df_aux, on="Bioma", how="left")
print(df_final2)

             system:index     AREA_KM2     Bioma  CD_Bioma  CD_UF  FID_BR_UF_  \
0    0000000000000000002a   903208.361  Amazônia         1     51          22   
1    00000000000000000029   329651.496  Amazônia         1     21           5   
2    0000000000000000002b   237754.172  Amazônia         1     11          25   
3    0000000000000000002d   164173.429  Amazônia         1     12           0   
4    0000000000000000002e  1559255.881  Amazônia         1     13           1   
..                    ...          ...       ...       ...    ...         ...   
898  00000000000000000024  1245870.704   Cerrado         3     15           2   
899  00000000000000000025   277423.627   Cerrado         3     17           4   
900  0000000000000000001e   586513.983   Cerrado         3     31          14   
901  00000000000000000021   248219.485   Cerrado         3     35          17   
902  00000000000000000022   199298.981   Cerrado         3     41          18   

     FID_lm_bio     NM_REGI

In [ ]:
unique_sources = df_final2["emission_source"].unique()
print(unique_sources)


['def' 'sl' 'fire']


In [ ]:
df_final2["source"] = df_final2["emission_source"].map({"def": "Deforestation", "sl": "Degradation", "fire": "Degradation"})


In [ ]:
df_final2["sum_fc"] = df_final2["sum"] * df_final2["fc"]
print(df_final2[["Bioma", "sum", "fc", "sum_fc"]].head())

      Bioma            sum     fc         sum_fc
0  Amazônia  486942.639563  1.013  493272.893878
1  Amazônia   82224.965385  1.013   83293.889935
2  Amazônia  549228.647120  1.013  556368.619532
3  Amazônia  132647.773896  1.013  134372.194956
4  Amazônia  561138.889319  1.013  568433.694880


In [ ]:
print(df_final2)

             system:index  AREA_KM2     Bioma  CD_Bioma  CD_UF  FID_BR_UF_  \
0    0000000000000000002a   903,208  Amazônia         1     51          22   
1    00000000000000000029   329,651  Amazônia         1     21           5   
2    0000000000000000002b   237,754  Amazônia         1     11          25   
3    0000000000000000002d   164,173  Amazônia         1     12           0   
4    0000000000000000002e 1,559,256  Amazônia         1     13           1   
..                    ...       ...       ...       ...    ...         ...   
898                     1   903,208  Pantanal         6     51          22   
899                     0   357,142  Pantanal         6     50          21   
900                     1   903,208  Pantanal         6     51          22   
901                     0   357,142  Pantanal         6     50          21   
902                     1   903,208  Pantanal         6     51          22   

     FID_lm_bio     NM_REGIAO               NM_UF SIGLA_UF  ...

# Table 1 - Estimated emissions and results (difference between FREL and the analyzed year) by biome for 2022 and 2023. The percentages in parentheses indicate that positive values represent emissions above the FREL 2017-2021 baseline, while negative values represent emissions below the FREL.

In [ ]:
# Converter a coluna 'ano' para numérico
df_final2['year'] = pd.to_numeric(df_final2['year'])

# Filtrar os anos de 2017 a 2021 e apenas 'co2' no type_gas
df_filtered = df_final2[df_final2['year'].between(2017, 2021) & (df_final2['type_gas'] == 'co2')]

# Agrupar por bioma e calcular a soma
tbl1 = df_filtered.groupby('Bioma', 'source', as_index=False)['sum'].sum()  # Use mean() se quiser média

# Renomear colunas
tbl1.rename(columns={'Bioma': 'Biome', 'sum': 'FREL'}, inplace=True)

# Configurar a exibição de floats sem notação científica
pd.options.display.float_format = '{:,.0f}'.format

# Exibir a tabela
tbl1


ValueError: No axis named source for object type DataFrame

In [ ]:
# Converter a coluna 'year' para numérico
df_final2['year'] = pd.to_numeric(df_final2['year'])

# Filtrar os anos de 2017 a 2021 e apenas 'co2' no type_gas
df_filtered = df_final2[(df_final2['year'].between(2017, 2021)) & (df_final2['type_gas'] == 'co2')]

# Agrupar por Bioma e Source e calcular a soma
tbl1 = df_filtered.groupby([ 'year', 'source' ], as_index=False)['sum'].sum()  # Use mean() se quiser média

# Renomear colunas
tbl1.rename(columns={'Bioma': 'Biome', 'sum': 'FREL'}, inplace=True)

# Configurar a exibição de floats sem notação científica
pd.options.display.float_format = '{:,.0f}'.format

# Exibir a tabela
print(tbl1)


   year         source        FREL
0  2017  Deforestation 511,298,786
1  2017    Degradation  39,263,992
2  2018  Deforestation 521,191,139
3  2018    Degradation  14,633,069
4  2019  Deforestation 680,542,751
5  2019    Degradation  30,414,388
6  2020  Deforestation 668,268,766
7  2020    Degradation  55,965,335
8  2021  Deforestation 791,938,330
9  2021    Degradation  36,368,598


In [ ]:
# Converter a coluna 'year' para numérico
df_final2['year'] = pd.to_numeric(df_final2['year'])

# Filtrar os anos de 2017 a 2021 e apenas 'co2' no type_gas
df_filtered = df_final2[(df_final2['year'].between(2017, 2021)) & (df_final2['type_gas'] == 'co2')]

# Agrupar por 'year' e calcular a soma de 'sum' e 'sum_fc'
tbl1 = df_filtered.groupby(['Bioma','year'], as_index=False)[['sum', 'sum_fc']].sum()  # Use mean() se quiser média

# Renomear colunas
tbl1.rename(columns={'sum': 'FREL'}, inplace=True)

# Configurar a exibição de floats sem notação científica
pd.options.display.float_format = '{:,.0f}'.format

# Exibir a tabela
print(tbl1)


NameError: name 'pd' is not defined

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Configurar o estilo
sns.set_style("whitegrid")

# Agrupar por year e calcular a soma total
df_agg = df_filtered.groupby('year', as_index=False)['sum_fc'].sum()
df_agg['sum_milhoes'] = df_agg['sum_fc'] / 1e6  # Converter para milhões

# Converter 'year' para numérico
df_agg['year'] = pd.to_numeric(df_agg['year'])

# Média de 2017 a 2021
media_2017_2021 = df_agg[(df_agg['year'] >= 2017) & (df_agg['year'] <= 2021)]['sum_milhoes'].mean()

# Criar a figura
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=df_agg, x='year', y='sum_milhoes', hue='year', palette='Reds', legend=False)

# Adicionar rótulos dentro das barras
for p in ax.patches:
    ax.annotate(f"{p.get_height():,.0f}",
                (p.get_x() + p.get_width() / 2, p.get_height() - 30),
                ha='center', va='top', fontsize=10, color='white', fontweight='bold')

# Adicionar linha tracejada para a média
plt.axhline(media_2017_2021, linestyle='--', color='black', linewidth=1.5)

# Adicionar texto alinhado à esquerda como legenda
plt.text(df_agg['year'].min() - 0.4, media_2017_2021,
         f'Média: {media_2017_2021:,.0f}', fontsize=11, color='black',
         fontweight='bold', ha='left', va='bottom', bbox=dict(facecolor='black', alpha=0.7, edgecolor='none'))

# Ajustes nos eixos e rótulos
plt.xlabel('')
plt.ylabel('tons CO₂ eq (milhões)')
plt.ylim(0, df_agg['sum_milhoes'].max() * 1.1)
plt.xticks(rotation=45)

# Adicionar legenda
plt.legend(['Official National FREL'], loc='upper left')

# Exibir o gráfico
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Configurar o estilo
sns.set_style("whitegrid")

# Agrupar por year e calcular a soma total
df_agg = df_final2.groupby('year', as_index=False)['sum_fc'].sum()
df_agg['sum_milhoes'] = df_agg['sum_fc'] / 1e6  # Converter para milhões

# Convert 'year' column to numeric
df_agg['year'] = pd.to_numeric(df_agg['year'])

# Média de 2017 a 2023
media_2017_2023 = df_agg[(df_agg['year'] >= 2017) & (df_agg['year'] <= 2021)]['sum_milhoes'].mean()

# Criar a figura
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=df_agg, x='year', y='sum_milhoes', hue='year', palette='Reds', legend=False)

# Adicionar rótulos dentro das barras
for p in ax.patches:
    ax.annotate(f"{p.get_height():,.0f}",
                (p.get_x() + p.get_width() / 2, p.get_height() - 30),
                ha='center', va='top', fontsize=10, color='white', fontweight='bold')

# Adicionar linha tracejada para a média
plt.axhline(media_2017_2023, linestyle='--', color='black', linewidth=1.5)
plt.text(df_agg['year'].min(), media_2017_2023 + 20,
         f'{media_2017_2023:,.0f}', fontsize=11, color='black', fontweight='bold')

# Ajustes nos eixos e rótulos
plt.xlabel('')
plt.ylabel('tons CO₂ eq (milhões)')
plt.ylim(0, df_agg['sum_milhoes'].max() * 1.1)
plt.xticks(rotation=45)

# Adicionar legenda
plt.legend(['Official National FREL'], loc='upper left')

# Exibir o gráfico
plt.show()